# Notebook 1 — Ingesta de Datos y Auditoría de Calidad 🔍
**TFG: Detección de Intrusiones a Gran Escala utilizando ML Distribuido y Big Data**

**Autor:** Eduardo Morillas Rodríguez

**Objetivo:** Cargar todos los CSVs del dataset CSE-CIC-IDS 2018, auditar la calidad,
**intentar recuperar labels inválidos** mediante un pipeline de normalización progresivo,
y solo eliminar las filas que sean verdaderamente irrecuperables.

**Filosofía: Recuperar primero, eliminar después.**

```
Label crudo → Trim → Espacios → Chars especiales → Case matching → Variantes → Fuzzy
                                                                             ↓
                                                                  ¿Coincide con VALID_LABELS?
                                                                   SÍ → ✅ Conservar
                                                                   NO → 🗑️ Eliminar
```

**Entrada:** CSVs en `DATA_RAW_PATH`  ·  **Salida:** Parquet en `DATA_PARQUET_PATH`

---
## Imports y Configuración

In [1]:
import glob
import re
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, FloatType, StringType
from pyspark.sql import types as T
import pandas as pd


from config import *


spark = get_spark_session("TFG_IDS_NB1_Ingesta")


print(f"📋 Configuración:")
print(f"   DATA_RAW_PATH:          {DATA_RAW_PATH}")
print(f"   DATA_PARQUET_PATH:      {DATA_PARQUET_PATH}")
print(f"   VALID_LABELS:           {len(VALID_LABELS)} clases reconocidas")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/05/03 18:53:27 WARN Utils: Your hostname, eespcachcpro01, resolves to a loopback address: 127.0.1.1; using 10.151.52.2 instead (on interface ens3f0)
26/05/03 18:53:27 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 18:53:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/05/03 18:53:27 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).


📋 Configuración:
   DATA_RAW_PATH:          /home/hpc/22231088student/tfg_ids/data/raw
   DATA_PARQUET_PATH:      /home/hpc/22231088student/tfg_ids/data/parquet/cse_cic_ids_2018
   VALID_LABELS:           15 clases reconocidas


---
## 1.1 — Carga de Todos los CSVs

In [2]:
csv_files = sorted(glob.glob(os.path.join(DATA_RAW_PATH, "*.csv")))


print(f"📁 Archivos CSV encontrados: {len(csv_files)}")
if len(csv_files) == 0:
    raise FileNotFoundError(f"No se encontraron CSVs en {DATA_RAW_PATH}.")


for f in csv_files:
    size_mb = os.path.getsize(f) / (1024**2)
    print(f"  📄 {os.path.basename(f)} — {size_mb:.1f} MB")

📁 Archivos CSV encontrados: 10
  📄 Friday-02-03-2018_TrafficForML_CICFlowMeter.csv — 336.0 MB
  📄 Friday-16-02-2018_TrafficForML_CICFlowMeter.csv — 318.3 MB
  📄 Friday-23-02-2018_TrafficForML_CICFlowMeter.csv — 365.1 MB
  📄 Thuesday-20-02-2018_TrafficForML_CICFlowMeter-002.csv — 3867.1 MB
  📄 Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv — 102.8 MB
  📄 Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv — 358.5 MB
  📄 Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv — 364.9 MB
  📄 Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv — 341.6 MB
  📄 Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv — 313.7 MB
  📄 Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv — 199.6 MB


In [3]:
dfs = []
for csv_file in csv_files:
    filename = os.path.basename(csv_file)
    df_temp = (
        spark.read.csv(csv_file, header=True, inferSchema=True)
        .withColumn("source_file", F.lit(filename))
    )
    dfs.append(df_temp)
    print(f"  ✅ {filename}: {df_temp.count():,} filas, {len(df_temp.columns)} cols")


df_all = dfs[0]
for df_temp in dfs[1:]:
    df_all = df_all.unionByName(df_temp, allowMissingColumns=True)


total_rows = df_all.count()
total_cols = len(df_all.columns)
print(f"\n📊 DATASET COMPLETO: {total_rows:,} filas × {total_cols} columnas")

  ✅ Friday-02-03-2018_TrafficForML_CICFlowMeter.csv: 1,048,575 filas, 81 cols


  ✅ Friday-16-02-2018_TrafficForML_CICFlowMeter.csv: 1,048,575 filas, 81 cols


  ✅ Friday-23-02-2018_TrafficForML_CICFlowMeter.csv: 1,048,575 filas, 81 cols


  ✅ Thuesday-20-02-2018_TrafficForML_CICFlowMeter-002.csv: 7,948,748 filas, 85 cols
  ✅ Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv: 331,125 filas, 81 cols


  ✅ Thursday-15-02-2018_TrafficForML_CICFlowMeter.csv: 1,048,575 filas, 81 cols


  ✅ Thursday-22-02-2018_TrafficForML_CICFlowMeter.csv: 1,048,575 filas, 81 cols


  ✅ Wednesday-14-02-2018_TrafficForML_CICFlowMeter.csv: 1,048,575 filas, 81 cols


  ✅ Wednesday-21-02-2018_TrafficForML_CICFlowMeter.csv: 1,048,575 filas, 81 cols
  ✅ Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv: 613,104 filas, 81 cols


[Stage 50:=================================>                   (513 + 66) / 808]


📊 DATASET COMPLETO: 16,233,002 filas × 85 columnas


In [4]:
print("📋 Esquema del DataFrame:")
df_all.printSchema()

📋 Esquema del DataFrame:
root
 |-- Dst Port: integer (nullable = true)
 |-- Protocol: integer (nullable = true)
 |-- Timestamp: string (nullable = true)
 |-- Flow Duration: long (nullable = true)
 |-- Tot Fwd Pkts: integer (nullable = true)
 |-- Tot Bwd Pkts: integer (nullable = true)
 |-- TotLen Fwd Pkts: double (nullable = true)
 |-- TotLen Bwd Pkts: double (nullable = true)
 |-- Fwd Pkt Len Max: double (nullable = true)
 |-- Fwd Pkt Len Min: double (nullable = true)
 |-- Fwd Pkt Len Mean: double (nullable = true)
 |-- Fwd Pkt Len Std: double (nullable = true)
 |-- Bwd Pkt Len Max: double (nullable = true)
 |-- Bwd Pkt Len Min: double (nullable = true)
 |-- Bwd Pkt Len Mean: double (nullable = true)
 |-- Bwd Pkt Len Std: double (nullable = true)
 |-- Flow Byts/s: double (nullable = true)
 |-- Flow Pkts/s: double (nullable = true)
 |-- Flow IAT Mean: double (nullable = true)
 |-- Flow IAT Std: double (nullable = true)
 |-- Flow IAT Max: double (nullable = true)
 |-- Flow IAT Min: doub

---
## 1.2 — Perfilado Estadístico Inicial

In [5]:
print("📊 Estadísticos descriptivos (primeras 10 columnas):")
df_all.select(df_all.columns[:10]).describe().show(vertical=True)

📊 Estadísticos descriptivos (primeras 10 columnas):


26/05/03 18:53:52 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 53:====================================================>(795 + 13) / 808]

-RECORD 0-------------------------------
 summary         | count                
 Dst Port        | 16232943             
 Protocol        | 16232943             
 Timestamp       | 16233002             
 Flow Duration   | 16232943             
 Tot Fwd Pkts    | 16232943             
 Tot Bwd Pkts    | 16232943             
 TotLen Fwd Pkts | 16232943             
 TotLen Bwd Pkts | 16232943             
 Fwd Pkt Len Max | 16232943             
 Fwd Pkt Len Min | 16232943             
-RECORD 1-------------------------------
 summary         | mean                 
 Dst Port        | 9164.072933971369    
 Protocol        | 8.754113471599082    
 Timestamp       | NULL                 
 Flow Duration   | 1.1813800896990892E7 
 Tot Fwd Pkts    | 23.533190931551967   
 Tot Bwd Pkts    | 6.312705835288154    
 TotLen Fwd Pkts | 973.0372564605199    
 TotLen Bwd Pkts | 4730.9314237104145   
 Fwd Pkt Len Max | 200.7620361261664    
 Fwd Pkt Len Min | 11.075998480374137   
-RECORD 2-------

In [6]:
print("🔍 Valores nulos por columna:")
print("=" * 60)


null_counts = df_all.select(
    [F.count(F.when(F.isnull(c), c)).alias(c) for c in df_all.columns]
).toPandas().T
null_counts.columns = ["null_count"]
null_counts["null_pct"] = (null_counts["null_count"] / total_rows * 100).round(2)
null_counts_sorted = null_counts[null_counts["null_count"] > 0].sort_values("null_count", ascending=False)


if len(null_counts_sorted) > 0:
    print(null_counts_sorted.to_string())
else:
    print("  ✅ No se encontraron valores nulos.")

🔍 Valores nulos por columna:


[Stage 56:===================================================> (789 + 19) / 808]

                   null_count  null_pct
Src IP                8284254     51.03
Flow ID               8284254     51.03
Dst IP                8284254     51.03
Src Port              8284254     51.03
Tot Bwd Pkts               59      0.00
TotLen Fwd Pkts            59      0.00
Flow Duration              59      0.00
Tot Fwd Pkts               59      0.00
Fwd Pkt Len Min            59      0.00
Fwd Pkt Len Mean           59      0.00
Fwd Pkt Len Std            59      0.00
Bwd Pkt Len Max            59      0.00
Bwd Pkt Len Min            59      0.00
Bwd Pkt Len Mean           59      0.00
TotLen Bwd Pkts            59      0.00
Fwd Pkt Len Max            59      0.00
Dst Port                   59      0.00
Protocol                   59      0.00
Flow IAT Mean              59      0.00
Flow Pkts/s                59      0.00
Flow Byts/s                59      0.00
Bwd Pkt Len Std            59      0.00
Flow IAT Std               59      0.00
Flow IAT Max               59      0.00


In [7]:
print("\n🔍 Duplicados exactos...")
cols_for_dup = [c for c in df_all.columns if c != "source_file"]
df_dup = df_all.groupBy(cols_for_dup).count().filter("count > 1")
n_dup_groups = df_dup.count()
n_dup_rows = df_dup.select(F.sum("count")).first()[0] if n_dup_groups > 0 else 0
print(f"  Grupos duplicados:   {n_dup_groups:,}")
if total_rows > 0 and n_dup_rows:
    print(f"  Filas en duplicados: {n_dup_rows:,} ({n_dup_rows/total_rows*100:.2f}%)")


🔍 Duplicados exactos...


[Stage 67:===================================>                  (84 + 44) / 128]

  Grupos duplicados:   107,671
  Filas en duplicados: 518,436 (3.19%)


In [8]:
print("\n🔍 Valores infinitos...")
numeric_cols = [
    field.name for field in df_all.schema.fields
    if isinstance(field.dataType, (DoubleType, FloatType))
]
inf_counts = {}
for col_name in numeric_cols:
    inf_count = df_all.filter(
        F.col(col_name).isin([float("inf"), float("-inf")])
    ).count()
    if inf_count > 0:
        inf_counts[col_name] = inf_count


if inf_counts:
    print(f"  ⚠️ Infinitos en {len(inf_counts)} columnas:")
    for col_name, count in sorted(inf_counts.items(), key=lambda x: -x[1]):
        print(f"    {col_name}: {count:,}")
else:
    print("  ✅ No se encontraron valores infinitos.")


🔍 Valores infinitos...


[Stage 203:====================>                               (153 + 64) / 391]

  ⚠️ Infinitos en 2 columnas:
    Flow Pkts/s: 95,760
    Flow Byts/s: 36,039


In [9]:
print("\n🔍 Columnas con varianza cero...")
zero_var_cols = []
for col_name in numeric_cols:
    n_distinct = df_all.select(col_name).distinct().count()
    if n_distinct <= 1:
        zero_var_cols.append(col_name)


if zero_var_cols:
    print(f"  ⚠️ Varianza cero ({len(zero_var_cols)}):")
    for c in zero_var_cols:
        print(f"    - {c}")
else:
    print("  ✅ Todas las columnas numéricas tienen varianza > 0.")


🔍 Columnas con varianza cero...


[Stage 470:=================================================>  (764 + 44) / 808]

  ✅ Todas las columnas numéricas tienen varianza > 0.


---
## 1.3 — Detección Automática de la Columna Label

In [10]:
label_col_original = None
for possible_name in ["Label", " Label", "label", " label"]:
    if possible_name in df_all.columns:
        label_col_original = possible_name
        break


if label_col_original is None:
    raise ValueError(f"❌ No se encontró columna Label. Columnas: {df_all.columns}")


print(f"✅ Columna Label detectada: '{label_col_original}'")


if label_col_original != "Label":
    df_all = df_all.withColumnRenamed(label_col_original, "Label")
    print(f"   Renombrada '{label_col_original}' → 'Label'")


# Guardar copia del label original ANTES de cualquier transformación
df_all = df_all.withColumn("Label_raw", F.col("Label"))


# Mostrar valores crudos
unique_raw = df_all.select("Label_raw").distinct().orderBy("Label_raw").toPandas()
print(f"\n📋 Labels CRUDOS encontrados ({len(unique_raw)}):")
for _, row in unique_raw.iterrows():
    val = repr(row["Label_raw"])  # repr para ver espacios, tabs, etc.
    status = "✅" if str(row["Label_raw"]).strip() in VALID_LABELS else "❓"
    print(f"  {status} {val}")

✅ Columna Label detectada: 'Label'



📋 Labels CRUDOS encontrados (16):
  ✅ 'Benign'
  ✅ 'Bot'
  ✅ 'Brute Force -Web'
  ✅ 'Brute Force -XSS'
  ❓ 'DDOS attack-HOIC'
  ❓ 'DDOS attack-LOIC-UDP'
  ✅ 'DDoS attacks-LOIC-HTTP'
  ✅ 'DoS attacks-GoldenEye'
  ✅ 'DoS attacks-Hulk'
  ✅ 'DoS attacks-SlowHTTPTest'
  ✅ 'DoS attacks-Slowloris'
  ✅ 'FTP-BruteForce'
  ✅ 'Infilteration'
  ❓ 'Label'
  ✅ 'SQL Injection'
  ✅ 'SSH-Bruteforce'


---
## 1.4 — Pipeline de Normalización de Labels

Se aplican **6 pasos progresivos** para intentar recuperar labels inválidos.
Después de cada paso se cuenta cuántos labels pasan de inválido a válido.

| Paso | Acción | Ejemplo |
|---|---|---|
| 1 | Trim (espacios laterales) | `" Benign "` → `"Benign"` |
| 2 | Colapsar espacios múltiples | `"Brute  Force"` → `"Brute Force"` |
| 3 | Eliminar caracteres especiales | `"Bot\t"` → `"Bot"` |
| 4 | Case-insensitive matching | `"benign"` → `"Benign"` |
| 5 | Mapeo de variantes conocidas | `"DDOS attack-LOIC-UDP"` → `"DDoS attack-LOIC-UDP"` |
| 6 | Fuzzy matching (sin espacios/guiones) | `"BruteForce-Web"` → `"Brute Force -Web"` |

In [11]:
# === Función para contar válidos e inválidos ===
def count_valid_invalid(df, label_col="Label"):
    n_valid = df.filter(F.col(label_col).isin(VALID_LABELS)).count()
    n_invalid = df.filter(~F.col(label_col).isin(VALID_LABELS)).count()
    return n_valid, n_invalid


# Estado inicial (antes de cualquier limpieza)
n_valid_initial, n_invalid_initial = count_valid_invalid(df_all, "Label_raw")
print(f"📊 ESTADO INICIAL:")
print(f"   Labels válidos:   {n_valid_initial:,}")
print(f"   Labels inválidos: {n_invalid_initial:,}")
print(f"   Total:            {total_rows:,}")


recovery_log = []

[Stage 487:======================================>             (600 + 64) / 808]

📊 ESTADO INICIAL:
   Labels válidos:   15,545,201
   Labels inválidos: 687,801
   Total:            16,233,002


### Paso 1: Trim (espacios laterales)

In [12]:
df_all = df_all.withColumn("Label", F.trim(F.col("Label")))


n_valid, n_invalid = count_valid_invalid(df_all)
recovered = n_valid - n_valid_initial
recovery_log.append({"paso": "1. Trim", "recuperados": recovered, "validos": n_valid, "invalidos": n_invalid})
print(f"  Paso 1 — Trim: {recovered:,} labels recuperados → {n_valid:,} válidos, {n_invalid:,} inválidos")

[Stage 493:=======================================>            (617 + 64) / 808]

  Paso 1 — Trim: 0 labels recuperados → 15,545,201 válidos, 687,801 inválidos


### Paso 2: Colapsar espacios múltiples

In [13]:
df_all = df_all.withColumn("Label", F.regexp_replace(F.col("Label"), r"\s+", " "))


n_valid, n_invalid = count_valid_invalid(df_all)
recovered = n_valid - recovery_log[-1]["validos"]
recovery_log.append({"paso": "2. Colapsar espacios", "recuperados": recovered, "validos": n_valid, "invalidos": n_invalid})
print(f"  Paso 2 — Colapsar espacios: {recovered:,} recuperados → {n_valid:,} válidos, {n_invalid:,} inválidos")

[Stage 499:================================================>   (753 + 55) / 808]

  Paso 2 — Colapsar espacios: 0 recuperados → 15,545,201 válidos, 687,801 inválidos


### Paso 3: Eliminar caracteres especiales (tabs, newlines, non-printable)

In [14]:
df_all = df_all.withColumn("Label", F.regexp_replace(F.col("Label"), r"[\t\r\n\x00-\x1f]", ""))
df_all = df_all.withColumn("Label", F.trim(F.col("Label")))  # Re-trim tras eliminar chars


n_valid, n_invalid = count_valid_invalid(df_all)
recovered = n_valid - recovery_log[-1]["validos"]
recovery_log.append({"paso": "3. Chars especiales", "recuperados": recovered, "validos": n_valid, "invalidos": n_invalid})
print(f"  Paso 3 — Chars especiales: {recovered:,} recuperados → {n_valid:,} válidos, {n_invalid:,} inválidos")

[Stage 505:====================================================>(801 + 7) / 808]

  Paso 3 — Chars especiales: 0 recuperados → 15,545,201 válidos, 687,801 inválidos


### Paso 4: Case-insensitive matching

In [15]:
# Crear diccionario: lowercase → forma canónica
case_map = {label.lower(): label for label in VALID_LABELS}


# Broadcast como variable y aplicar con UDF
case_map_broadcast = spark.sparkContext.broadcast(case_map)


@F.udf(StringType())
def fix_case(label):
    if label is None:
        return None
    mapping = case_map_broadcast.value
    lower_label = label.lower().strip()
    return mapping.get(lower_label, label)  # Si no encuentra, devuelve el original


df_all = df_all.withColumn("Label", fix_case(F.col("Label")))


n_valid, n_invalid = count_valid_invalid(df_all)
recovered = n_valid - recovery_log[-1]["validos"]
recovery_log.append({"paso": "4. Case-insensitive", "recuperados": recovered, "validos": n_valid, "invalidos": n_invalid})
print(f"  Paso 4 — Case matching: {recovered:,} recuperados → {n_valid:,} válidos, {n_invalid:,} inválidos")

[Stage 511:===================================================>(795 + 13) / 808]

  Paso 4 — Case matching: 687,742 recuperados → 16,232,943 válidos, 59 inválidos


### Paso 5: Mapeo de variantes conocidas

In [16]:
# Variantes explícitas detectadas en la documentación y common errors
variant_mapping = {
    # Variantes de DDoS (mayúsculas)
    "DDOS attack-LOIC-UDP":     "DDoS attack-LOIC-UDP",
    "DDOS attack-HOIC":         "DDoS attack-HOIC",
    "DDOS attacks-LOIC-HTTP":   "DDoS attacks-LOIC-HTTP",
    # Variantes de Brute Force (guion sin espacio)
    "Brute Force-Web":          "Brute Force -Web",
    "Brute Force-XSS":          "Brute Force -XSS",
    "BruteForce-Web":           "Brute Force -Web",
    "BruteForce-XSS":           "Brute Force -XSS",
    "Brute Force - Web":        "Brute Force -Web",
    "Brute Force - XSS":        "Brute Force -XSS",
    # Variantes de SSH/FTP (mayúsculas/minúsculas)
    "SSH-BruteForce":           "SSH-Bruteforce",
    "SSH-Brute Force":          "SSH-Bruteforce",
    "SSH-bruteforce":           "SSH-Bruteforce",
    "FTP-BruteForce":           "FTP-BruteForce",
    "FTP-Brute Force":          "FTP-BruteForce",
    "FTP-bruteforce":           "FTP-BruteForce",
    # Typos comunes en Infiltration
    "Infiltration":             "Infilteration",
    "infilteration":            "Infilteration",
    "Infiltering":              "Infilteration",
    # DoS variantes
    "DOS attacks-GoldenEye":    "DoS attacks-GoldenEye",
    "DOS attacks-Hulk":         "DoS attacks-Hulk",
    "DOS attacks-SlowHTTPTest": "DoS attacks-SlowHTTPTest",
    "DOS attacks-Slowloris":    "DoS attacks-Slowloris",
}


applied_variants = []
for variant, canonical in variant_mapping.items():
    n_match = df_all.filter(F.col("Label") == variant).count()
    if n_match > 0:
        df_all = df_all.withColumn(
            "Label",
            F.when(F.col("Label") == variant, canonical).otherwise(F.col("Label"))
        )
        applied_variants.append((variant, canonical, n_match))
        print(f"  🔄 '{variant}' → '{canonical}' ({n_match:,} filas)")


if not applied_variants:
    print("  ℹ️ No se encontraron variantes conocidas.")


n_valid, n_invalid = count_valid_invalid(df_all)
recovered = n_valid - recovery_log[-1]["validos"]
recovery_log.append({"paso": "5. Variantes conocidas", "recuperados": recovered, "validos": n_valid, "invalidos": n_invalid})
print(f"\n  Paso 5 — Variantes: {recovered:,} recuperados → {n_valid:,} válidos, {n_invalid:,} inválidos")

  🔄 'FTP-BruteForce' → 'FTP-BruteForce' (193,360 filas)


[Stage 583:===============================================>    (744 + 64) / 808]


  Paso 5 — Variantes: 0 recuperados → 16,232,943 válidos, 59 inválidos


### Paso 6: Fuzzy matching (último recurso)

Se compara eliminando espacios y guiones. Si coincide con algún label
válido, se asigna. Es el paso más agresivo — se documenta cada corrección.

In [17]:
# Crear diccionario: forma "limpia" (sin espacios, sin guiones, lowercase) → forma canónica
fuzzy_map = {}
for label in VALID_LABELS:
    clean_key = re.sub(r"[\s\-_]", "", label).lower()
    fuzzy_map[clean_key] = label


fuzzy_map_broadcast = spark.sparkContext.broadcast(fuzzy_map)
valid_labels_broadcast = spark.sparkContext.broadcast(set(VALID_LABELS))


@F.udf(StringType())
def fuzzy_fix(label):
    if label is None:
        return None
    valid = valid_labels_broadcast.value
    if label in valid:
        return label  # Ya es válido, no tocar
    mapping = fuzzy_map_broadcast.value
    clean_key = re.sub(r"[\s\-_]", "", label).lower()
    return mapping.get(clean_key, label)  # Si no encuentra, devuelve original


df_all = df_all.withColumn("Label", fuzzy_fix(F.col("Label")))


n_valid, n_invalid = count_valid_invalid(df_all)
recovered = n_valid - recovery_log[-1]["validos"]
recovery_log.append({"paso": "6. Fuzzy matching", "recuperados": recovered, "validos": n_valid, "invalidos": n_invalid})
print(f"  Paso 6 — Fuzzy: {recovered:,} recuperados → {n_valid:,} válidos, {n_invalid:,} inválidos")

[Stage 589:==================================================> (783 + 25) / 808]

  Paso 6 — Fuzzy: 0 recuperados → 16,232,943 válidos, 59 inválidos


### 📊 Resumen del Pipeline de Normalización

In [18]:
recovery_df = pd.DataFrame(recovery_log)
recovery_df["pct_validos"] = (recovery_df["validos"] / total_rows * 100).round(2)


print("\n" + "=" * 70)
print("📊 RESUMEN DEL PIPELINE DE NORMALIZACIÓN")
print("=" * 70)
print(recovery_df.to_string(index=False))


total_recovered = n_valid - n_valid_initial
print(f"\n  Labels ANTES del pipeline:  {n_valid_initial:,} válidos / {n_invalid_initial:,} inválidos")
print(f"  Labels DESPUÉS del pipeline: {n_valid:,} válidos / {n_invalid:,} inválidos")
print(f"  Total RECUPERADOS:           {total_recovered:,}")
if n_invalid_initial > 0:
    print(f"  Tasa de recuperación:        {total_recovered/n_invalid_initial*100:.1f}%")


📊 RESUMEN DEL PIPELINE DE NORMALIZACIÓN
                  paso  recuperados  validos  invalidos  pct_validos
               1. Trim            0 15545201     687801        95.76
  2. Colapsar espacios            0 15545201     687801        95.76
   3. Chars especiales            0 15545201     687801        95.76
   4. Case-insensitive       687742 16232943         59       100.00
5. Variantes conocidas            0 16232943         59       100.00
     6. Fuzzy matching            0 16232943         59       100.00

  Labels ANTES del pipeline:  15,545,201 válidos / 687,801 inválidos
  Labels DESPUÉS del pipeline: 16,232,943 válidos / 59 inválidos
  Total RECUPERADOS:           687,742
  Tasa de recuperación:        100.0%


---
## 1.5 — Auditoría de Calidad por Archivo (Post-Normalización)

In [19]:
print("📋 AUDITORÍA DE CALIDAD POR ARCHIVO (tras normalización)")
print("=" * 90)


audit_by_file = (
    df_all
    .groupBy("source_file")
    .agg(
        F.count("*").alias("total_filas"),
        F.sum(
            F.when(F.col("Label").isin(VALID_LABELS), 1).otherwise(0)
        ).alias("labels_validos"),
    )
    .withColumn("labels_invalidos", F.col("total_filas") - F.col("labels_validos"))
    .withColumn("pct_corrupcion", F.round(F.col("labels_invalidos") / F.col("total_filas") * 100, 2))
    .orderBy(F.desc("pct_corrupcion"))
)


audit_by_file.show(50, truncate=False)


audit_pd = audit_by_file.toPandas()
audit_csv_path = os.path.join(FIGURES_PATH, "auditoria_calidad_por_archivo.csv")
audit_pd.to_csv(audit_csv_path, index=False)
print(f"💾 Tabla de auditoría guardada: {audit_csv_path}")


files_with_issues = audit_pd[audit_pd["pct_corrupcion"] > 0]
files_warning = audit_pd[audit_pd["pct_corrupcion"] > WARNING_CORRUPTION_PCT]


print(f"\n📋 RESUMEN:")
print(f"  Archivos limpios (0% inválidos):                   {len(audit_pd) - len(files_with_issues)}")
print(f"  Archivos con algún label inválido restante:        {len(files_with_issues)}")
print(f"  Archivos con ⚠️ alta corrupción (>{WARNING_CORRUPTION_PCT}%): {len(files_warning)}")


if len(files_warning) > 0:
    for _, row in files_warning.iterrows():
        print(f"    ⚠️ {row['source_file']}: {row['pct_corrupcion']}% ({int(row['labels_invalidos']):,} filas)")

📋 AUDITORÍA DE CALIDAD POR ARCHIVO (tras normalización)


+-----------------------------------------------------+-----------+--------------+----------------+--------------+
|source_file                                          |total_filas|labels_validos|labels_invalidos|pct_corrupcion|
+-----------------------------------------------------+-----------+--------------+----------------+--------------+
|Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv    |331125     |331100        |25              |0.01          |
|Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv   |613104     |613071        |33              |0.01          |
|Friday-02-03-2018_TrafficForML_CICFlowMeter.csv      |1048575    |1048575       |0               |0.0           |
|Friday-16-02-2018_TrafficForML_CICFlowMeter.csv      |1048575    |1048574       |1               |0.0           |
|Friday-23-02-2018_TrafficForML_CICFlowMeter.csv      |1048575    |1048575       |0               |0.0           |
|Thuesday-20-02-2018_TrafficForML_CICFlowMeter-002.csv|7948748    |7948748      

[Stage 595:=================================================>  (768 + 40) / 808]

💾 Tabla de auditoría guardada: /home/hpc/22231088student/tfg_ids/figures/auditoria_calidad_por_archivo.csv

📋 RESUMEN:
  Archivos limpios (0% inválidos):                   8
  Archivos con algún label inválido restante:        2
  Archivos con ⚠️ alta corrupción (>10%): 0


---
## 1.6 — Análisis de Labels Irrecuperables

In [20]:
df_still_invalid = df_all.filter(~F.col("Label").isin(VALID_LABELS))
n_still_invalid = df_still_invalid.count()


if n_still_invalid > 0:
    print(f"🔴 LABELS IRRECUPERABLES: {n_still_invalid:,} filas ({n_still_invalid/total_rows*100:.2f}%)")
    print("=" * 70)
    print("   Estos labels NO pudieron ser mapeados a ninguna clase válida")
    print("   tras 6 pasos de normalización.\n")


    # ¿Qué valores son?
    irrecoverable_dist = (
        df_still_invalid
        .groupBy("Label").count()
        .orderBy(F.desc("count"))
    ).toPandas()


    print(f"📊 Valores irrecuperables ({len(irrecoverable_dist)}):")
    for _, row in irrecoverable_dist.iterrows():
        print(f"  ❌ '{row['Label']}' — {int(row['count']):,} filas")


    # Comparar con label original para entender qué pasó
    print(f"\n📊 Desglose por archivo:")
    irrecoverable_by_file = (
        df_still_invalid
        .groupBy("source_file", "Label").count()
        .orderBy("source_file", F.desc("count"))
    ).toPandas()


    for filename in irrecoverable_by_file["source_file"].unique():
        file_data = irrecoverable_by_file[irrecoverable_by_file["source_file"] == filename]
        total_inv = file_data["count"].sum()
        print(f"\n  📄 {filename} ({total_inv:,} filas irrecuperables):")
        for _, row in file_data.iterrows():
            print(f"     ❌ '{row['Label']}' — {int(row['count']):,}")
else:
    print("✅ TODOS los labels fueron recuperados o ya eran válidos.")
    print("   No hay filas que eliminar por labels inválidos.")

🔴 LABELS IRRECUPERABLES: 59 filas (0.00%)
   Estos labels NO pudieron ser mapeados a ninguna clase válida
   tras 6 pasos de normalización.



📊 Valores irrecuperables (1):
  ❌ 'Label' — 59 filas

📊 Desglose por archivo:



  📄 Friday-16-02-2018_TrafficForML_CICFlowMeter.csv (1 filas irrecuperables):
     ❌ 'Label' — 1

  📄 Thursday-01-03-2018_TrafficForML_CICFlowMeter.csv (25 filas irrecuperables):
     ❌ 'Label' — 25

  📄 Wednesday-28-02-2018_TrafficForML_CICFlowMeter.csv (33 filas irrecuperables):
     ❌ 'Label' — 33


---
## 1.7 — Eliminación de Filas Irrecuperables

In [29]:
if n_still_invalid > 0:
    print("🧹 Eliminando filas con labels irrecuperables...")

    df_clean = df_all.filter(F.col("Label").isin(VALID_LABELS))
    n_after = df_clean.count()
    n_removed = total_rows - n_after

    print(f"\n📊 RESULTADO:")
    print(f"  Filas originales:    {total_rows:,}")
    print(f"  Filas recuperadas:   {total_recovered:,} (por normalización)")
    print(f"  Filas eliminadas:    {n_removed:,} (irrecuperables)")
    print(f"  Filas conservadas:   {n_after:,} ({n_after/total_rows*100:.2f}%)")

    # Verificar que no se perdieron clases
    classes_after = set(df_clean.select("Label").distinct().toPandas()["Label"].tolist())
    classes_lost = set(VALID_LABELS) - classes_after

    if classes_lost:
        print(f"\n  ⚠️ Clases que ya no aparecen: {classes_lost}")
    else:
        print(f"\n  ✅ Las {len(classes_after)} clases siguen representadas.\n")

    # -------------------------------------------------------------------------
    # GENERACIÓN DINÁMICA DE LA CAJA ASCII PARA EVITAR DESAJUSTES
    # -------------------------------------------------------------------------
    box_width = 71  # Ancho interno de la caja
    
    # Definimos el contenido línea por línea
    box_lines = [
        "",
        "  Pipeline de normalización aplicado (6 pasos):",
        "  - Trim, colapsar espacios, caracteres especiales",
        "  - Case-insensitive matching, variantes conocidas, fuzzy matching",
        "",
        "  Resultado:",
        f"  - Labels recuperados: {total_recovered:,}",
        f"  - Labels irrecuperables: {n_removed:,} → eliminados",
        f"  - Filas conservadas: {n_after:,} ({n_after/total_rows*100:.1f}%)",
        "",
        "  Las filas eliminadas contienen labels que NO pertenecen al",
        "  catálogo oficial (15 clases) ni a ninguna variante reconocible.",
        "  No se pueden usar para clasificación supervisada.",
        ""
    ]

    # Imprimimos la caja asegurando que cada línea mida exactamente 'box_width'
    print(f"╔{'═' * box_width}╗")
    print(f"║{'DECISIÓN: ELIMINAR FILAS IRRECUPERABLES'.center(box_width)}║")
    print(f"╠{'═' * box_width}╣")
    
    for line in box_lines:
        # El formato :<{box_width} alinea el texto a la izquierda y rellena con espacios
        print(f"║{line:<{box_width}}║")
        
    print(f"╚{'═' * box_width}╝")

else:
    df_clean = df_all
    n_after = total_rows
    n_removed = 0
    print("ℹ️ No hay filas que eliminar. Todas tienen labels válidos.")

# Eliminar columna temporal
df_clean = df_clean.drop("Label_raw")

🧹 Eliminando filas con labels irrecuperables...



📊 RESULTADO:
  Filas originales:    16,233,002
  Filas recuperadas:   687,742 (por normalización)
  Filas eliminadas:    59 (irrecuperables)
  Filas conservadas:   16,232,943 (100.00%)


[Stage 659:====================================================>(807 + 1) / 808]


  ✅ Las 15 clases siguen representadas.

╔═══════════════════════════════════════════════════════════════════════╗
║                DECISIÓN: ELIMINAR FILAS IRRECUPERABLES                ║
╠═══════════════════════════════════════════════════════════════════════╣
║                                                                       ║
║  Pipeline de normalización aplicado (6 pasos):                        ║
║  - Trim, colapsar espacios, caracteres especiales                     ║
║  - Case-insensitive matching, variantes conocidas, fuzzy matching     ║
║                                                                       ║
║  Resultado:                                                           ║
║  - Labels recuperados: 687,742                                        ║
║  - Labels irrecuperables: 59 → eliminados                             ║
║  - Filas conservadas: 16,232,943 (100.0%)                             ║
║                                                                     

---
## 1.8 — Distribución Final y Guardado en Parquet

In [22]:
print("📊 DISTRIBUCIÓN FINAL DE CLASES")
print("=" * 70)


final_dist = (
    df_clean
    .groupBy("Label").count()
    .withColumn("pct", F.round(F.col("count") / F.lit(n_after) * 100, 4))
    .orderBy(F.desc("count"))
)


final_dist.show(truncate=False)

📊 DISTRIBUCIÓN FINAL DE CLASES


[Stage 628:==================================================> (782 + 26) / 808]

+------------------------+--------+------+
|Label                   |count   |pct   |
+------------------------+--------+------+
|Benign                  |13484708|83.07 |
|DDoS attack-HOIC        |686012  |4.226 |
|DDoS attacks-LOIC-HTTP  |576191  |3.5495|
|DoS attacks-Hulk        |461912  |2.8455|
|Bot                     |286191  |1.763 |
|FTP-BruteForce          |193360  |1.1912|
|SSH-Bruteforce          |187589  |1.1556|
|Infilteration           |161934  |0.9976|
|DoS attacks-SlowHTTPTest|139890  |0.8618|
|DoS attacks-GoldenEye   |41508   |0.2557|
|DoS attacks-Slowloris   |10990   |0.0677|
|DDoS attack-LOIC-UDP    |1730    |0.0107|
|Brute Force -Web        |611     |0.0038|
|Brute Force -XSS        |230     |0.0014|
|SQL Injection           |87      |5.0E-4|
+------------------------+--------+------+



In [23]:
print(f"💾 Guardando datos en Parquet...")
print(f"   Ruta: {DATA_PARQUET_PATH}")
print(f"   Particionado por: Label")
print(f"   Compresión: snappy")


df_to_save = df_clean.drop("source_file")
df_to_save.write.mode("overwrite").partitionBy("Label").parquet(DATA_PARQUET_PATH)


print(f"✅ Datos guardados correctamente.")

💾 Guardando datos en Parquet...
   Ruta: /home/hpc/22231088student/tfg_ids/data/parquet/cse_cic_ids_2018
   Particionado por: Label
   Compresión: snappy


26/05/03 19:05:30 WARN DAGScheduler: Broadcasting large task binary with size 1497.0 KiB
                                                                                

✅ Datos guardados correctamente.


In [24]:
from pathlib import Path


print("\n" + "=" * 70)
print("📋 RESUMEN FINAL — NOTEBOOK 1")
print("=" * 70)
print(f"  Archivos CSV procesados:    {len(csv_files)}")
print(f"  Filas originales:           {total_rows:,}")
print(f"  Labels recuperados:         {total_recovered:,} (normalización)")
print(f"  Filas eliminadas:           {n_removed:,} (irrecuperables)")
print(f"  Filas finales:              {n_after:,}")
print(f"  Columnas:                   {len(df_to_save.columns)}")
print(f"  Labels únicos:              {df_clean.select('Label').distinct().count()}")
if applied_variants:
    print(f"  Variantes corregidas:       {len(applied_variants)}")
print(f"  Formato de salida:          Parquet (snappy)")
print(f"  Ruta:                       {DATA_PARQUET_PATH}")


if os.path.exists(DATA_PARQUET_PATH):
    total_size = sum(f.stat().st_size for f in Path(DATA_PARQUET_PATH).rglob("*") if f.is_file())
    print(f"  Tamaño en disco:            {total_size / (1024**3):.2f} GB")


print(f"\n💡 TIP: Eliminar CSVs de {DATA_RAW_PATH} para liberar espacio.")
print(f"\n➡️  Siguiente: Notebook 2 — EDA")


📋 RESUMEN FINAL — NOTEBOOK 1
  Archivos CSV procesados:    10
  Filas originales:           16,233,002
  Labels recuperados:         687,742 (normalización)
  Filas eliminadas:           59 (irrecuperables)
  Filas finales:              16,232,943
  Columnas:                   84


[Stage 632:==================================================> (792 + 16) / 808]

  Labels únicos:              15
  Variantes corregidas:       1
  Formato de salida:          Parquet (snappy)
  Ruta:                       /home/hpc/22231088student/tfg_ids/data/parquet/cse_cic_ids_2018
  Tamaño en disco:            2.05 GB

💡 TIP: Eliminar CSVs de /home/hpc/22231088student/tfg_ids/data/raw para liberar espacio.

➡️  Siguiente: Notebook 2 — EDA


In [30]:
spark.stop()



